# 🧩 Reto 4 – Integración final de resultados

**Objetivo:** combinar los resultados de los retos anteriores para obtener un diagnóstico integral de la calidad del agua.

### 🔹 Instrucciones
- Importa los resultados obtenidos en los notebooks anteriores.
- Integra los análisis (pH, temperatura, oxígeno) en un solo DataFrame.
- Realiza una conclusión general del estado del agua.

In [5]:
# Tu código aquí

#import pandas as pd
#datos = pd.read_csv('../data/4_resultados_agua.csv')

In [11]:
# ============================================================
# 🧩 Reto 4 – Integración final de resultados
# ============================================================

# Importamos pandas, que es la librería para manejar tablas de datos
import pandas as pd

# ── PASO 1: CARGAR LOS DATOS ─────────────────────────────────
# pd.read_csv lee el archivo CSV y lo convierte en una tabla (DataFrame)
# El resultado lo guardamos en la variable "df"
df = pd.read_csv("../data/4_resultados_agua.csv")

# ── PASO 2: VER LAS PRIMERAS FILAS ───────────────────────────
# .head() nos muestra las primeras 5 filas de la tabla
# Sirve para verificar que los datos cargaron bien
print("=== Primeras filas ===")
print(df.head())

# ── PASO 3: VER LA ESTRUCTURA ────────────────────────────────
# .info() nos dice cuántas filas hay, qué columnas existen
# y qué tipo de dato tiene cada columna (número, texto, etc.)
print("\n=== Estructura del DataFrame ===")
print(df.info())

# ── PASO 4: ESTADÍSTICAS BÁSICAS ─────────────────────────────
# .describe() calcula automáticamente el promedio, mínimo,
# máximo y otros valores de todas las columnas numéricas
print("\n=== Estadísticas básicas ===")
print(df.describe())

# ── PASO 5: CLASIFICAR EL pH ─────────────────────────────────
# Creamos una función que recibe un número (el valor de pH)
# y devuelve una etiqueta según el rango en que esté

def clasificar_ph(valor):
    # Si el pH es menor a 6.5, el agua es ácida
    if valor < 6.5:
        return "Ácido"
    # Si el pH está entre 6.5 y 8.5, el agua es neutra (normal)
    elif valor <= 8.5:
        return "Neutro"
    # Si el pH es mayor a 8.5, el agua es básica
    else:
        return "Básico"

# .apply() aplica la función a cada fila de la columna "pH"
# y guarda el resultado en una nueva columna llamada "categoria_ph"
df["categoria_ph"] = df["pH"].apply(clasificar_ph)

# ── PASO 6: CREAR ALERTAS POR VARIABLE ───────────────────────
# Para cada variable ambiental creamos una columna de alerta
# que dice "Alerta" si el valor está fuera del rango normal,
# o "Normal" si está dentro del rango aceptable

# Alerta de pH: fuera del rango entre 6.5 y 8.5
df["ph_alerta"] = df["pH"].apply(
    lambda x: "Alerta" if x < 6.5 or x > 8.5 else "Normal"
)

# Alerta de temperatura: mayor a 25 grados
df["temp_alerta"] = df["Temp"].apply(
    lambda x: "Alerta" if x > 25 else "Normal"
)

# Alerta de oxígeno: menor a 6 mg/L es bajo y preocupante
df["oxigeno_alerta"] = df["OD_promedio"].apply(
    lambda x: "Alerta" if x < 6 else "Normal"
)

# ── PASO 7: DIAGNÓSTICO INTEGRAL POR MUESTRA ─────────────────
# Esta función revisa las 3 alertas de cada fila
# y decide qué tan grave es el estado del agua en ese punto

def diagnostico_integral(fila):
    # Contamos cuántas alertas tiene esa fila
    cantidad_alertas = sum([
        fila["ph_alerta"] == "Alerta",       # 1 si hay alerta de pH
        fila["temp_alerta"] == "Alerta",     # 1 si hay alerta de temperatura
        fila["oxigeno_alerta"] == "Alerta"   # 1 si hay alerta de oxígeno
    ])

    # Según la cantidad de alertas, asignamos un diagnóstico
    if cantidad_alertas == 0:
        return "✅ Apta"           # Sin problemas
    elif cantidad_alertas == 1:
        return "⚠️ Observación"   # Un problema, hay que vigilar
    else:
        return "❌ No apta"        # Dos o más problemas, agua en mal estado

# Aplicamos la función fila por fila con axis=1
df["diagnostico"] = df.apply(diagnostico_integral, axis=1)

# ── PASO 8: PROMEDIOS POR SITIO ───────────────────────────────
# .groupby("Punto") agrupa todas las filas que tienen el mismo sitio
# .mean() calcula el promedio de cada grupo
# .round(2) redondea a 2 decimales para que sea más legible
print("\n=== Promedios por sitio ===")
resumen = df.groupby("Punto")[["pH", "Temp", "OD_promedio"]].mean().round(2)
print(resumen)

# ── PASO 9: MOSTRAR EL DATAFRAME FINAL ───────────────────────
# Mostramos solo las columnas más importantes para ver
# el resultado de toda la integración
print("\n=== Tabla integrada (primeras 10 filas) ===")
print(df[["Punto", "pH", "Temp", "OD_promedio",
          "categoria_ph", "diagnostico"]].head(10))

# ── PASO 10: CONCLUSIÓN GENERAL ───────────────────────────────
# Contamos cuántas muestras cayeron en cada diagnóstico
print("\n=== Conteo de diagnósticos ===")
print(df["diagnostico"].value_counts())

# Calculamos los totales para el resumen final
total       = len(df)
aptas       = (df["diagnostico"] == "✅ Apta").sum()
observacion = (df["diagnostico"] == "⚠️ Observación").sum()
no_aptas    = (df["diagnostico"] == "❌ No apta").sum()

# Contamos cuántas muestras hay en cada categoría de pH
# Recordar los rangos:
#   🔴 Ácido  → pH < 6.5
#   🟢 Neutro → 6.5 ≤ pH ≤ 8.5
#   🔵 Básico → pH > 8.5
acido   = (df["categoria_ph"] == "Ácido").sum()
neutro  = (df["categoria_ph"] == "Neutro").sum()
basico  = (df["categoria_ph"] == "Básico").sum()

# Mostramos el resumen con porcentajes de diagnóstico
print("\n=== Conclusión general del estado del agua ===")
print(f"Total de muestras analizadas : {total}")
print(f"  ✅ Aptas para consumo       : {aptas}  ({aptas/total*100:.1f}%)")
print(f"  ⚠️  En observación          : {observacion}  ({observacion/total*100:.1f}%)")
print(f"  ❌ No aptas                 : {no_aptas}  ({no_aptas/total*100:.1f}%)")

# Mostramos el resumen con porcentajes por categoría de pH
print("\n=== Distribución por categoría de pH ===")
print(f"Total de muestras analizadas : {total}")
print(f"  🔴 Ácido  (pH < 6.5)        : {acido}  ({acido/total*100:.1f}%)")
print(f"  🟢 Neutro (6.5 ≤ pH ≤ 8.5) : {neutro}  ({neutro/total*100:.1f}%)")
print(f"  🔵 Básico (pH > 8.5)        : {basico}  ({basico/total*100:.1f}%)")

=== Primeras filas ===
  Punto   Temp    pH  OD_promedio
0     A  22.25  7.15          6.2
1     B  24.30  7.20          5.8
2     C  19.80  6.85          4.9
3     D  23.65  7.05          6.5
4     E  25.15  7.25          5.0

=== Estructura del DataFrame ===
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Punto        5 non-null      str    
 1   Temp         5 non-null      float64
 2   pH           5 non-null      float64
 3   OD_promedio  5 non-null      float64
dtypes: float64(3), str(1)
memory usage: 292.0 bytes
None

=== Estadísticas básicas ===
            Temp        pH  OD_promedio
count   5.000000  5.000000     5.000000
mean   23.030000  7.100000     5.680000
std     2.093621  0.158114     0.712039
min    19.800000  6.850000     4.900000
25%    22.250000  7.050000     5.000000
50%    23.650000  7.150000     5.800000
75%    24.300000  7.200000    

In [8]:
# ============================================================
# 🧩 Reto 4 – Integración final de resultados
# Objetivo: combinar los resultados de los retos anteriores
# para obtener un diagnóstico integral de la calidad del agua.
# ============================================================

%pip install pandas
import pandas as pd

# ── 1. CARGAR DATOS ──────────────────────────────────────────
df = pd.read_csv("../data/4_resultados_agua.csv")
# ── 2. VER PRIMERAS FILAS ────────────────────────────────────
print("=== Primeras filas ===")
print(df.head())

# ── 3. VER ESTRUCTURA ────────────────────────────────────────
print("\n=== Estructura del DataFrame ===")
print(df.info())

# ── 4. ESTADÍSTICAS BÁSICAS ──────────────────────────────────
print("\n=== Estadísticas básicas ===")
print(df.describe())

# ── 5. CLASIFICACIÓN DE pH (del Reto 3) ──────────────────────
def clasificar_ph(valor):
    if valor < 6.5:
        return "Ácido"
    elif valor <= 8.5:
        return "Neutro"
    else:
        return "Básico"

df["categoria_ph"] = df["pH"].apply(clasificar_ph)

# ── 6. FILTROS (del Reto 2) ───────────────────────────────────
# Condición 1: pH fuera del rango neutro
df["ph_alerta"] = df["pH"].apply(lambda x: "Alerta" if x < 6.5 or x > 8.5 else "Normal")

# Condición 2: Temperatura alta (> 25°C)
df["temp_alerta"] = df["Temp"].apply(lambda x: "Alerta" if x > 25 else "Normal")

# Condición 3: Oxígeno bajo (< 6 mg/L)
df["oxigeno_alerta"] = df["OD_promedio"].apply(lambda x: "Alerta" if x < 6 else "Normal")

# ── 7. DIAGNÓSTICO INTEGRAL ───────────────────────────────────
def diagnostico_integral(row):
    alertas = sum([
        row["ph_alerta"] == "Alerta",
        row["temp_alerta"] == "Alerta",
        row["oxigeno_alerta"] == "Alerta"
    ])
    if alertas == 0:
        return "✅ Apta"
    elif alertas == 1:
        return "⚠️ Observación"
    else:
        return "❌ No apta"

df["diagnostico"] = df.apply(diagnostico_integral, axis=1)

# ── 8. RESUMEN POR SITIO (del Reto 3 – groupby) ───────────────
print("\n=== Promedios por sitio ===")
resumen = df.groupby("Punto")[["pH", "Temp", "OD_promedio"]].mean().round(2)
print(resumen)

# ── 9. CONCLUSIÓN GENERAL ─────────────────────────────────────
print("\n=== DataFrame integrado (primeras filas) ===")
print(df[["Punto", "pH", "Temp", "OD_promedio",
          "categoria_ph", "diagnostico"]].head(10))

print("\n=== Conteo de diagnósticos ===")
print(df["diagnostico"].value_counts())

print("\n=== Conclusión general ===")
total = len(df)
aptas = (df["diagnostico"] == "✅ Apta").sum()
no_aptas = (df["diagnostico"] == "❌ No apta").sum()
observacion = (df["diagnostico"] == "⚠️ Observación").sum()

print(f"Total de muestras analizadas : {total}")
print(f"  ✅ Aptas para consumo       : {aptas}  ({aptas/total*100:.1f}%)")
print(f"  ⚠️  En observación          : {observacion}  ({observacion/total*100:.1f}%)")
print(f"  ❌ No aptas                 : {no_aptas}  ({no_aptas/total*100:.1f}%)")

Note: you may need to restart the kernel to use updated packages.
=== Primeras filas ===
  Punto   Temp    pH  OD_promedio
0     A  22.25  7.15          6.2
1     B  24.30  7.20          5.8
2     C  19.80  6.85          4.9
3     D  23.65  7.05          6.5
4     E  25.15  7.25          5.0

=== Estructura del DataFrame ===
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Punto        5 non-null      str    
 1   Temp         5 non-null      float64
 2   pH           5 non-null      float64
 3   OD_promedio  5 non-null      float64
dtypes: float64(3), str(1)
memory usage: 292.0 bytes
None

=== Estadísticas básicas ===
            Temp        pH  OD_promedio
count   5.000000  5.000000     5.000000
mean   23.030000  7.100000     5.680000
std     2.093621  0.158114     0.712039
min    19.800000  6.850000     4.900000
25%    22.250000  7.050000     5.000000
50% 